In [6]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu, spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# =============================
# Paths
# =============================
DATA_DIR = Path("C:/Users/ldesa/PycharmProjects/TAP basic/notebooks/test")
OUT_TABLES = DATA_DIR / "output" / "tables"
OUT_FIG = DATA_DIR / "output" / "figures"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIG.mkdir(parents=True, exist_ok=True)

# =============================
# Style (pastel colors)
# =============================
PALETTE = {
    "Non-Expert": "#AEC6CF",  # pastel blue
    "Expert": "#FFB347"       # pastel orange
}
GROUP_ORDER = ["Non-Expert", "Expert"]
sns.set_theme(style="whitegrid")

# =============================
# Load data
# =============================
tasks = pd.read_csv(DATA_DIR / "tasks.csv")
survey = pd.read_csv(DATA_DIR / "survey.csv")

# =============================
# SUS computation
# =============================
SUS_ITEMS = {
    1: "1. Penso che vorrei usare questo sistema frequentemente.",
    2: "2. Ho trovato il sistema inutilmente complesso.",
    3: "3. Ho ritenuto il sistema facile da usare.",
    4: "4. Credo che avrei bisogno del supporto di un tecnico per poter utilizzare questo sistema.",
    5: "5. Ho trovato che le varie funzioni del sistema sono ben integrate.",
    6: "6. Penso che ci sia stata troppa incoerenza nel sistema.",
    7: "7. Immagino che la maggior parte delle persone imparerebbe a usare questo sistema molto rapidamente.",
    8: "8. Ho trovato il sistema molto macchinoso da usare.",
    9: "9. Mi sono sentito molto sicuro nell’utilizzare il sistema.",
    10: "10. Ho dovuto imparare molte cose prima di riuscire a utilizzare il sistema."
}
SUS_POS = [1, 3, 5, 7, 9]
SUS_NEG = [2, 4, 6, 8, 10]

def compute_sus(row):
    pos = sum(row[SUS_ITEMS[i]] - 1 for i in SUS_POS)
    neg = sum(5 - row[SUS_ITEMS[i]] for i in SUS_NEG)
    return (pos + neg) * 2.5

survey["SUS"] = survey.apply(compute_sus, axis=1)

# =============================
# NASA-TLX
# =============================
TLX_MAP = {
    "MentalDemand": "Domanda Mentale",
    "PhysicalDemand": "Domanda Fisica",
    "TemporalDemand": "Domanda Temporale",
    "Performance": "Prestazione",
    "Effort": "Sforzo",
    "Frustration": "Frustrazione"
}
for k, v in TLX_MAP.items():
    survey[k] = survey[v]

survey["TLX"] = survey[list(TLX_MAP.keys())].mean(axis=1)

# =============================
# Background variables
# =============================
survey = survey.rename(columns={
    "User ID (anonimo)": "user_id",
    "Età": "age",
    "Livello di istruzione": "education",
    "Competenze informatiche (1 = basse, 5 = alte)": "computer_skills",
    "Conoscenza della piattaforma IFTTT (1 = nulla, 5 = avanzata)": "ifttt_familiarity"
})

EDU_MAP = {
    "Scuola superiore": 1,
    "Laurea triennale": 2,
    "Laurea magistrale": 3,
    "Master/Specializzazione": 4,
    "Dottorato": 5
}
survey["education_level"] = survey["education"].map(EDU_MAP)

survey["user_group"] = np.where(
    (survey["computer_skills"] >= 4) | (survey["ifttt_familiarity"] >= 3),
    "Expert",
    "Non-Expert"
)

# =============================
# Task-level aggregation
# =============================
task_user = (
    tasks.groupby(["user_id", "task_id"])
    .agg(
        attempts=("attempt", "max"),
        final_success=("eval_correct_bool", "max"),
        first_attempt_success=("eval_correct_bool", "first")
    ).reset_index()
)

user_perf = (
    task_user.groupby("user_id")
    .agg(
        mean_attempts=("attempts", "mean"),
        success_rate=("final_success", "mean"),
        first_attempt_rate=("first_attempt_success", "mean")
    ).reset_index()
)

df = survey.merge(user_perf, on="user_id")

# =============================
# Descriptive table
# =============================
group_desc = df.groupby("user_group").agg(
    n=("user_id", "count"),
    SUS=("SUS", "mean"),
    TLX=("TLX", "mean"),
    Attempts=("mean_attempts", "mean"),
    Success=("success_rate", "mean")
)
group_desc.to_latex(
    OUT_TABLES / "group_descriptive_statistics.tex",
    float_format="%.2f",
    caption="Descriptive statistics by user group.",
    label="tab:group-descriptives"
)

# =============================
# Background → outcome correlations (Spearman)
# =============================
BACKGROUND = ["age", "education_level", "computer_skills", "ifttt_familiarity"]
OUTCOMES = ["SUS", "TLX", "mean_attempts", "success_rate",
            "MentalDemand", "TemporalDemand", "Effort", "Frustration"]

rows = []
for g in ["Non-Expert", "Expert"]:
    sub = df[df.user_group == g]
    for b in BACKGROUND:
        for o in OUTCOMES:
            aligned = sub[[b, o]].dropna()
            if len(aligned) >= 5:
                rho, p = spearmanr(aligned[b], aligned[o])
                rows.append([g, b, o, rho, p, len(aligned)])

corr_df = pd.DataFrame(
    rows,
    columns=["group", "background", "outcome", "rho", "p_value", "n"]
)

corr_df.to_latex(
    OUT_TABLES / "background_vs_outcomes.tex",
    float_format="%.3f",
    caption="Within-group Spearman correlations between background variables and key outcomes.",
    label="tab:bg-outcomes"
)

# =============================
# Heatmap (overview)
# =============================
plt.figure(figsize=(10, 4))
sns.heatmap(
    corr_df.pivot_table(index="background", columns="outcome", values="rho"),
    cmap="RdBu_r", center=0, annot=True, fmt=".2f"
)
plt.title("Background Factors vs. Key Outcomes (Spearman ρ)")
plt.tight_layout()
plt.savefig(OUT_FIG / "background_vs_outcomes_heatmap.png", dpi=300)
plt.close()

# =============================
# Plots with pastel colors
# =============================
for y, fname, title, ylabel in [
    ("SUS", "sus_by_user_group.png", "Perceived Usability (SUS)", "SUS"),
    ("TLX", "tlx_by_user_group.png", "Overall Cognitive Workload (NASA-TLX)", "TLX"),
    ("mean_attempts", "attempts_by_user_group.png", "Task Iterations", "Attempts")
]:
    plt.figure()
    sns.boxplot(data=df, x="user_group", y=y,
                order=GROUP_ORDER, palette=PALETTE)
    plt.title(title)
    plt.xlabel("User Group")
    plt.ylabel(ylabel)
    plt.savefig(OUT_FIG / fname, dpi=300)
    plt.close()

tlx_long = df.melt(
    id_vars=["user_group"],
    value_vars=list(TLX_MAP.keys()),
    var_name="TLX Dimension",
    value_name="Score"
)

plt.figure(figsize=(8, 4))
sns.boxplot(
    data=tlx_long,
    x="TLX Dimension",
    y="Score",
    hue="user_group",
    palette=PALETTE
)
plt.xticks(rotation=30)
plt.title("NASA-TLX Dimensions by User Group")
plt.tight_layout()
plt.savefig(OUT_FIG / "tlx_dimensions_by_user_group.png", dpi=300)
plt.close()

print("✔ Complete analysis finished: pastel plots + background-to-behavior relations saved.")


C:\Users\ldesa\AppData\Local\Temp\ipykernel_14604\4103406622.py:191: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="user_group", y=y,
C:\Users\ldesa\AppData\Local\Temp\ipykernel_14604\4103406622.py:191: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="user_group", y=y,
C:\Users\ldesa\AppData\Local\Temp\ipykernel_14604\4103406622.py:191: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="user_group", y=y,


✔ Complete analysis finished: pastel plots + background-to-behavior relations saved.
